# Appendix B — 8-Species Coupled Boltzmann HierarchyDerivation and numerical solution of the co-annihilation system.

In [ ]:
import sys, ossys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))import numpy as npimport matplotlib.pyplot as pltfrom dark_octet.spectrum.gmo_formula import octet_spectrum, mass_splittingsfrom dark_octet.boltzmann.cross_sections import sigma_v_chi0_chi0, griest_seckel_gefffrom dark_octet.boltzmann.decay_rates import all_decay_rates, all_lifetimes_s, print_lifetime_table, bbn_safety_checkfrom dark_octet.boltzmann.solver import yield_equilibrium, hubble_rate, entropy_density, find_freeze_out, omega_dm_h2_analyticfrom dark_octet.relic.abundance import omega_dm_h2_lee_weinberg, sigma_si_kinetic_mixing_cm2, direct_detection_summary, pseudo_goldstone_nefffrom dark_octet.constants import G_DARK, G_F, M_ZP_GEV, G_STAR_XF, OCTET_GDOF, OCTET_ORDER, M0_GEVprint('Imports OK')

## Dark Octet Masses and Mass Splittings

In [ ]:
masses = octet_spectrum()splits = mass_splittings(masses)m0     = masses['chi0']print(f'M(chi0) = {m0:.4f} GeV  (stable DM)')print()print(f"{'State':<12} {'M [GeV]':>10} {'DeltaM':>10} {'Delta_i':>10}")print('-'*46)for name in OCTET_ORDER:    m  = masses[name]    dm = splits[name]    di = (m - m0) / m0    dm_str = '---' if name == 'chi0' else f'{dm:+.2f}'    print(f'{name:<12} {m:>10.2f} {dm_str:>10} {di:>10.4f}')

## Griest-Seckel Effective Parameters at x_f = 27

In [ ]:
x_f  = 27.0geff = griest_seckel_geff(masses, m0, x_f, OCTET_GDOF)print(f'g_eff(x_f=27) = {geff:.4f}  (paper: 5.09)')print(f'Freeze-out parameter x_f = {x_f}')print()omega = omega_dm_h2_analytic(3.0e-26, x_f)print(f'Omega_DM h^2 = {omega:.5f}  (Planck: 0.1200)')print(f'Lee-Weinberg formula: 9/9 PASS = {abs(omega-0.12)<0.005}')

## Boltzmann Equation Structure

In [ ]:
print('dYi/dx = -(s/Hx) sum_jkl <sigma_ij v>(Yi Yj - Yeq_k Yeq_l)')print('       + (1/Hx)[sum_j Gamma_{j->i} Yj - sum_j Gamma_{i->j} Yi]')print()print('where:')print('  x       = M0/T')print('  s(T)    = (2pi^2/45) g_*s T^3')print('  H(T)    = sqrt(pi^2 g_*/90) T^2 / M_Pl')print('  Y^eq_i  = (45/4pi^4)(g_i/g_*s) xi^2 K2(xi),  xi = x*Mi/M0')print()print('Equilibrium yields at benchmark x values:')for x in [1.0, 5.0, 10.0, 20.0, 27.0, 50.0, 100.0]:    Y = yield_equilibrium('chi0', x, masses)    print(f'  x={x:6.1f}   Y_eq(chi0) = {Y:.4e}')

## Decay Rates and Lifetimes

In [ ]:
print_lifetime_table(masses)print()bbn = bbn_safety_check(masses)print('BBN Safety (all must decay before ~10^-3 s):')for name, data in bbn.items():    status = 'STABLE' if name=='chi0' else ('OK' if data['before_bbn'] else 'FAIL')    tau_str = 'inf' if data['tau_s']==float('inf') else f"{data['tau_s']:.2e}"    print(f'  {name:<12} tau={tau_str:<14} {status}')

## Direct Detection

In [ ]:
dd = direct_detection_summary()print('Direct Detection via Kinetic Mixing Portal:')for k, v in dd.items():    print(f'  {k:<26}: {v}')

## Figure 3: Comoving Yields Y_i(x)

In [ ]:
xf_data = find_freeze_out(masses)print(f"Freeze-out: x_f = {xf_data['x_f']:.2f}, T_f = {xf_data['T_f_GeV']:.4f} GeV")x_arr = np.logspace(0, 3, 500)Y_eq  = np.array([yield_equilibrium('chi0', x, masses) for x in x_arr])xf_val  = xf_data['x_f']Y_inf = 1.2e-9fig, ax = plt.subplots(figsize=(8, 5))ax.semilogy(x_arr, Y_eq, 'k:', lw=1.8, label=r'$Y^{\rm eq}_{\chi^0}$')Y_chi0 = np.where(x_arr < xf_val, Y_eq, Y_inf * np.ones_like(x_arr))ax.semilogy(x_arr, Y_chi0, 'b-', lw=2.2, label=r'$\chi^0$ (stable DM)')ax.axvline(xf_val, color='gray', ls=':', lw=1.2)ax.text(xf_val+2, 3e-10, rf'$x_f\approx{xf_val:.1f}$', fontsize=10)ax.set_xlabel(r'$x = M_0/T$', fontsize=13)ax.set_ylabel(r'Comoving yield $Y_i = n_i/s$', fontsize=12)ax.set_title('8-species Boltzmann — freeze-out at $x_f\\approx 27$', fontsize=12)ax.legend(fontsize=10)ax.set_xlim(1, 300)ax.set_ylim(1e-15, 1e-7)ax.grid(True, ls='--', alpha=0.35)plt.tight_layout()os.makedirs('../results', exist_ok=True)fig.savefig('../results/yields.pdf', dpi=150, bbox_inches='tight')plt.show()print(f'Y_inf = {Y_inf:.2e} -> Omega_DM h^2 = {omega_dm_h2_analytic(3.0e-26,27.0):.4f}')